# 1.5 Build the 2026 childcare centre layer with Nursery fees

The supply side is every centre ECDA lists as operating in October 2026 (`CentreAttributes_2026.csv`,
1,875 rows), located as a point, with the Nursery (4 years old) Full Day fee for Singapore citizens
and its Master Plan subzone. This is the team's **canonical centre layer** (decision 2026-11-01,
`docs/decisions.md`): 1,796 centres across all service models (CC 1,467, KN 183, DS 84, EYC-DS 57,
EYC 5), replacing the stale 2021 GeoJSON and the CC/KN-only `data/Centres_2026.geojson`
(`../0_Data_Exploration/geocode_centres_onemap.ipynb`). Steps:

1. Drop the 80 placeholder rows whose `centre_code` is "na" (no fee can be joined and most have no
   name); they are audited in 1_06, and the one confirmed as an operating centre is added back here.
2. Locate each centre through the 2021 GeoJSON by postal code (a postal code is one building in
   Singapore); when several 2021 centres share the postal code the most similar name is used.
3. Geocode only the centres left unlocated (new, relocated or invalid postal code) with the OneMap
   search API, through a cache so reruns make no network calls.
4. Join the Nursery Full Day/SC fee by `centre_code` (Daryl's rules, now in `src/bt4015/fees.py`).
5. Attach the 2019 subzone, planning area and region.

Original script: `scripts/build_childcare_centres_2026.py` (Jessie), fee rules from
`scripts/join_childcare_fees.py` (Daryl). This run is served entirely from the OneMap cache
(`ALLOW_ONEMAP = False`); if a new centre row ever needs a lookup the notebook stops and says so.

**Inputs**
- `data/CentreAttributes_2026.csv` — ECDA centre list, Oct 2026.
- `data/ListingofCentreServices.csv` — ECDA fee listing by centre, level, service type, citizenship.
- `data/ChildCareServices_2021.geojson` — 2021 centre points, used for coordinates by postal code.
- `data/MasterPlanSubZones_2019.geojson`.
- `output/onemap_geocode_cache.csv` — OneMap responses keyed by query (shared with 1_06).

**Outputs**
- `output/childcare_centres_2026.gpkg`, layer `centres` — 1,796 centres, EPSG:3414, with
  `FULL_DAY_SC_NURSERY_FEE`, `fee_status`, `location_source`, `record_source`, subzone fields;
  read by 1_06, 1_07, 1_08 and every analysis notebook. (Layer `planned` in the same file is
  written by 1_06 and is left untouched here.)
- `output/childcare_centres_2026_review.csv` — centres that were geocoded or still lack a location
  or fee, for review.
- `output/onemap_geocode_cache.csv` — rewritten with the same content.

In [1]:
import sys
from pathlib import Path

import csv
import difflib
import os
import time
from collections import defaultdict

import geopandas as gpd
import pandas as pd
import requests

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

from bt4015.fees import BENCHMARK_LEVEL, core_name, load_fee_records

# changed 2026-11-10: paths through bt4015.paths (the script used paths relative to the repo root).
ATTRIBUTES_CSV = DATA / "CentreAttributes_2026.csv"
FEES_CSV = DATA / "ListingofCentreServices.csv"
CENTRES_2021 = DATA / "ChildCareServices_2021.geojson"
SUBZONES = DATA / "MasterPlanSubZones_2019.geojson"
OUT_GPKG = OUT / "childcare_centres_2026.gpkg"
OUT_LAYER = "centres"
OUT_REVIEW = OUT / "childcare_centres_2026_review.csv"
GEOCODE_CACHE = OUT / "onemap_geocode_cache.csv"

# "na" rows verified as operating on 2026-11-01 (see 1_06 / excluded_centres_verification_2026.csv),
# keyed by tp_code. The name comes from the operator's page, as the ECDA row has none.
VERIFIED_ADDITIONS = {
    "TP1356": "PCF Sparkletots Preschool @ Yuhua Blk 285C (KN)",
}

CRS_OUT = "EPSG:3414"      # SVY21 / Singapore TM, metres
ONEMAP_SEARCH = "https://www.onemap.gov.sg/api/common/elastic/search"
NAME_MATCH_MIN = 0.6       # below this a same-postal match is labelled postal_only
REQUEST_PAUSE_S = 0.5      # OneMap allows about 250 calls a minute
MAX_RETRIES = 5
CACHE_SAVE_EVERY = 25
ALLOW_ONEMAP = False       # added 2026-11-10: False = every lookup must come from the cache (no network)

## 1. Load the 2026 ECDA centre list

1,875 rows; 80 have `centre_code` "na" and are dropped here (they are the subject of 1_06). The
one "na" row verified as an operating centre, TP1356, is added back with `record_source =
"verified_addition"`; it keeps the ECDA fields as listed, with no centre code and therefore no fee.

In [2]:
# 1. Read everything as text (centre codes and postal codes are identifiers, not numbers).
centres = pd.read_csv(ATTRIBUTES_CSV, dtype=str, encoding="utf-8-sig").fillna("")
centres["postal_code"] = centres["postal_code"].str.strip().str.zfill(6)
placeholder = centres["centre_code"].str.strip().str.lower() == "na"
print(f"2026 rows: {len(centres)}, dropping {placeholder.sum()} with centre_code 'na'")

# 2. Add back the verified "na" row(s) under their operator-page name.
added = centres[placeholder & centres["tp_code"].isin(VERIFIED_ADDITIONS)].copy()
added["centre_name"] = added["tp_code"].map(VERIFIED_ADDITIONS)
added["centre_code"] = ""
added["record_source"] = "verified_addition"
print(f"Adding back {len(added)} verified 'na' rows: {', '.join(added['tp_code'])}")

# 3. Listed centres, one row per centre_code.
centres = centres[~placeholder].drop_duplicates("centre_code").assign(record_source="ecda_listed")
centres = pd.concat([centres, added], ignore_index=True)
centres = centres[
    ["centre_code", "tp_code", "record_source", "centre_name", "organisation_code",
     "organisation_description", "service_model", "centre_address", "postal_code", "last_updated"]
]
print(f"Centres to locate: {len(centres)}")
print(centres["service_model"].value_counts().to_string())

2026 rows: 1875, dropping 80 with centre_code 'na'
Adding back 1 verified 'na' rows: TP1356
Centres to locate: 1796
service_model
CC        1467
KN         183
DS          84
EYC-DS      57
EYC          5


## 2. Locate centres through the 2021 GeoJSON by postal code

A shared postal code means the same building, so a renamed centre at the same address keeps a
valid location. When several 2021 centres share the postal code, the one with the most similar
core name is taken; a similarity below `NAME_MATCH_MIN` is still a location match but is labelled
`2021_postal_only` so it can be reviewed.

In [3]:
def valid_postal(value: str) -> bool:
    """Six digits and not the 000000 placeholder."""
    value = (value or "").strip()
    return len(value) == 6 and value.isdigit() and value != "000000"


def name_similarity(a: str, b: str) -> float:
    """Sequence similarity of two centre names after stripping legal suffixes."""
    return difflib.SequenceMatcher(None, core_name(a), core_name(b)).ratio()


# 1. Index the 2021 points by (valid) postal code.
old = gpd.read_file(CENTRES_2021).to_crs("EPSG:4326")
old["postal"] = old["ADDRESSPOSTALCODE"].fillna("").astype(str).str.strip().str.zfill(6)
by_postal = defaultdict(list)
for index, postal in old["postal"].items():
    if valid_postal(postal):
        by_postal[postal].append(index)
print(f"2021 centres: {len(old)}, distinct valid postal codes: {len(by_postal)}")

# 2. For each 2026 centre, take the 2021 point at its postal code with the most similar name.
lon, lat, object_ids, sources, scores = [], [], [], [], []
for row in centres.itertuples():
    candidates = by_postal.get(row.postal_code, []) if valid_postal(row.postal_code) else []
    if not candidates:
        lon.append(None); lat.append(None); object_ids.append(None)
        sources.append(None); scores.append(None)
        continue
    best = max(candidates, key=lambda i: name_similarity(row.centre_name, old.at[i, "NAME"]))
    score = name_similarity(row.centre_name, old.at[best, "NAME"])
    point = old.geometry[best]
    lon.append(point.x); lat.append(point.y)
    object_ids.append(int(old.at[best, "OBJECTID"]))
    sources.append("2021_postal_name" if score >= NAME_MATCH_MIN else "2021_postal_only")
    scores.append(round(score, 3))

centres["lon"], centres["lat"] = lon, lat
centres["objectid_2021"] = pd.array(object_ids, dtype="Int64")
centres["location_source"] = sources
centres["name_similarity_2021"] = scores
print("Matched to 2021 GeoJSON:")
print(centres["location_source"].value_counts(dropna=False).to_string())

2021 centres: 1925, distinct valid postal codes: 1739


Matched to 2021 GeoJSON:
location_source
2021_postal_name    1224
NaN                  305
2021_postal_only     267


## 3. Geocode the rest from the OneMap cache

Each unlocated centre is tried first by postal code (accepting only a result in that postal code),
then by the first two parts of its address ("86,JALAN PERGAM,EAST VIEW GARDEN,488362" →
"86 JALAN PERGAM"). Every query made on 2026-11-01 is in `output/onemap_geocode_cache.csv`, so
this run reads 305 lookups from the cache and makes no API call. The cache is saved back after the
loop (and every 25 new queries when the API is allowed) so an interrupted run resumes.

In [4]:
def load_cache() -> dict:
    """Cache rows keyed by query: {query: {query, lat, lon, postal, address}}."""
    if not GEOCODE_CACHE.exists():
        return {}
    with GEOCODE_CACHE.open(encoding="utf-8", newline="") as handle:
        return {row["query"]: row for row in csv.DictReader(handle)}


def save_cache(cache: dict) -> None:
    GEOCODE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    with GEOCODE_CACHE.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["query", "lat", "lon", "postal", "address"])
        writer.writeheader()
        writer.writerows(cache.values())


def onemap_search(query: str, session: requests.Session) -> dict:
    """One OneMap search; a postal-code query only accepts a result in that postal code."""
    params = {"searchVal": query, "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": 1}
    headers = {"Authorization": os.environ["ONEMAP_TOKEN"]} if os.environ.get("ONEMAP_TOKEN") else {}
    for attempt in range(MAX_RETRIES):
        response = session.get(ONEMAP_SEARCH, params=params, headers=headers, timeout=30)
        if response.status_code != 429:
            break
        wait = float(response.headers.get("Retry-After") or 15 * (attempt + 1))
        print(f"  rate limited on {query!r}, waiting {wait:.0f}s")
        time.sleep(wait)
    response.raise_for_status()
    results = response.json().get("results", [])
    if query.isdigit():
        results = [r for r in results if r.get("POSTAL") == query]
    if not results:
        return {"query": query, "lat": "", "lon": "", "postal": "", "address": ""}
    top = results[0]
    return {"query": query, "lat": top["LATITUDE"], "lon": top["LONGITUDE"],
            "postal": top.get("POSTAL", ""), "address": top.get("ADDRESS", "")}


def address_query(address: str) -> str:
    """'86,JALAN PERGAM,EAST VIEW GARDEN,488362' -> '86 JALAN PERGAM'."""
    parts = [p.strip() for p in address.split(",") if p.strip()]
    parts = [p for p in parts if not (len(p) == 6 and p.isdigit())]
    return " ".join(parts[:2])


# 1. Which centres still need a location, and what the cache already holds.
cache = load_cache()
session = requests.Session()
todo = centres.index[centres["lon"].isna()]
print(f"Geocoding {len(todo)} unmatched centres with OneMap ({len(cache)} cached queries)")
new_queries = 0
try:
    for index in todo:
        row = centres.loc[index]
        queries = [row.postal_code] if valid_postal(row.postal_code) else []
        queries.append(address_query(row.centre_address))
        for query in filter(None, queries):
            if query not in cache:
                # changed 2026-11-10: a rerun must not call OneMap unnoticed. Every query of the
                # 2026-11-01 build is cached, so this only fires for new or edited centre rows.
                if not ALLOW_ONEMAP:
                    raise RuntimeError(f"{query!r} is not in the OneMap cache; set ALLOW_ONEMAP = True to query the API")
                cache[query] = onemap_search(query, session)
                new_queries += 1
                if new_queries % CACHE_SAVE_EVERY == 0:
                    save_cache(cache)
                    print(f"  {new_queries} queries done")
                time.sleep(REQUEST_PAUSE_S)
            # 2. First query with a hit wins: postal code before address.
            hit = cache[query]
            if hit["lat"]:
                centres.at[index, "lat"] = float(hit["lat"])
                centres.at[index, "lon"] = float(hit["lon"])
                centres.at[index, "location_source"] = (
                    "onemap_postal" if query == row.postal_code else "onemap_address"
                )
                break
finally:
    # Keep finished lookups even if the run crashes, so a rerun resumes.
    save_cache(cache)
print(f"New OneMap queries this run: {new_queries}")
print(centres["location_source"].value_counts(dropna=False).to_string())

Geocoding 305 unmatched centres with OneMap (346 cached queries)


New OneMap queries this run: 0
location_source
2021_postal_name    1224
onemap_postal        305
2021_postal_only     267


## 4. Join the Nursery Full Day/SC fee by centre code

`load_fee_records` keeps the fee rows for "Nursery (4 yrs old)", Full Day, SC, and gives a centre a
fee only when its rows agree on one most common value. `fee_status` records why a centre has none:
no listing for that level/service/citizenship, conflicting fees, or no centre code (the verified
addition).

In [5]:
# 1. One fee per centre_code where the listing is unambiguous (Daryl's rules, src/bt4015/fees.py).
records = {r.centre_code: r.fee for r in load_fee_records(FEES_CSV, ATTRIBUTES_CSV)}
print(f"Centres with a unique {BENCHMARK_LEVEL} Full Day/SC fee in the listing: {len(records):,}")

# 2. Centres that have any benchmark row at all, to tell "no listing" from "conflicting fees".
fees = pd.read_csv(FEES_CSV, dtype=str, encoding="utf-8-sig").fillna("")
benchmark = fees[
    (fees["type_of_service"].str.strip().str.casefold() == "full day")
    & (fees["type_of_citizenship"].str.strip().str.casefold() == "sc")
    & (fees["levels_offered"].str.strip() == BENCHMARK_LEVEL)
]
has_benchmark_row = set(benchmark["centre_code"])

# 3. Attach the fee and label the gaps.
centres["FULL_DAY_SC_NURSERY_FEE"] = centres["centre_code"].map(records)
centres["fee_status"] = "listed"
centres.loc[~centres["centre_code"].isin(has_benchmark_row), "fee_status"] = "no_nursery_full_day_sc_listing"
centres.loc[
    centres["centre_code"].isin(has_benchmark_row) & centres["FULL_DAY_SC_NURSERY_FEE"].isna(),
    "fee_status",
] = "conflicting_fees"
centres.loc[centres["centre_code"] == "", "fee_status"] = "no_centre_code"
print(centres["fee_status"].value_counts().to_string())
fee = centres["FULL_DAY_SC_NURSERY_FEE"].dropna()
print(f"Fee (S$/month): median {fee.median():.0f}, p10 {fee.quantile(.1):.0f}, p90 {fee.quantile(.9):.0f}, max {fee.max():.0f}")

Centres with a unique Nursery (4 yrs old) Full Day/SC fee in the listing: 1,598


fee_status
listed                            1587
no_nursery_full_day_sc_listing     193
conflicting_fees                    15
no_centre_code                       1
Fee (S$/month): median 650, p10 610, p90 2031, max 4390


## 5. Points in SVY21 with their subzone

Centres without any location are left out of the layer (and listed in the review file). A centre
can fall in at most one subzone, which the assertion checks.

In [6]:
# 1. Build the point layer from the located centres and project to metres.
located = centres["lon"].notna()
gdf = gpd.GeoDataFrame(
    centres[located],
    geometry=gpd.points_from_xy(centres.loc[located, "lon"], centres.loc[located, "lat"]),
    crs="EPSG:4326",
).to_crs(CRS_OUT)

# 2. Subzone, planning area and region by point-in-polygon.
zones = gpd.read_file(SUBZONES).to_crs(CRS_OUT)
cols = ["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "REGION_N"]
gdf = gpd.sjoin(gdf, zones[cols + ["geometry"]], how="left", predicate="within")
assert len(gdf) == len(centres[located]), "a centre fell in two subzones"
gdf = gdf.drop(columns="index_right")
gdf["centre_code"] = gdf["centre_code"].replace("", None)
print(f"Centres: {len(centres)}, located: {located.sum()}, no location: {(~located).sum()}")
print(f"Centres outside every subzone: {gdf['SUBZONE_C'].isna().sum()}")

Centres: 1796, located: 1796, no location: 0
Centres outside every subzone: 0


## 6. Write the layer and the review file, and check the headline counts

In [7]:
# 1. Layer "centres" only; layer "planned" (1_06) in the same GeoPackage is left as it is.
OUT_GPKG.parent.mkdir(parents=True, exist_ok=True)
gdf.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")

# 2. Review file: anything not located through the 2021 file, or without a fee.
review = centres[
    ~centres["location_source"].fillna("").str.startswith("2021")
    | centres["FULL_DAY_SC_NURSERY_FEE"].isna()
]
review.to_csv(OUT_REVIEW, index=False)

print(f"Wrote {OUT_GPKG.relative_to(ROOT)} layer {OUT_LAYER} ({len(gdf)} centres) and "
      f"{OUT_REVIEW.relative_to(ROOT)} ({len(review)} rows)")
print(gdf["record_source"].value_counts().to_string())
print("\nCentres by service model:")
print(gdf["service_model"].value_counts().to_string())
print(f"\nCentres with a listed fee: {(gdf['fee_status'] == 'listed').sum()}")

Wrote output/childcare_centres_2026.gpkg layer centres (1796 centres) and output/childcare_centres_2026_review.csv (420 rows)
record_source
ecda_listed          1795
verified_addition       1

Centres by service model:
service_model
CC        1467
KN         183
DS          84
EYC-DS      57
EYC          5

Centres with a listed fee: 1587
